# HandWrite AI — Handwritten Character Recognition

This notebook walks through loading **MNIST**, **EMNIST Letters**, **EMNIST ByClass**, and **USPS**, building a CNN for 36 classes (`0-9` and `A-Z`), training, and evaluating the model.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'training'))
from train_model import CLASS_NAMES, load_character_dataset, NUM_CLASSES

print('TensorFlow', tf.__version__)
print('Classes:', ''.join(CLASS_NAMES))

## 2. Dataset Loading and Exploration

In [ ]:
x_train, y_train, x_test, y_test = load_character_dataset()

print(f'Training images shape: {x_train.shape}')
print(f'Testing images shape: {x_test.shape}')
print(f'Number of classes: {NUM_CLASSES}')
print(f'Train class counts: {np.bincount(y_train, minlength=NUM_CLASSES)}')

### Class Distribution

In [ ]:
plt.figure(figsize=(14, 4))
sns.countplot(x=y_train, hue=y_train, palette='viridis', legend=False)
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, rotation=90)
plt.title('Class Distribution in Training Set')
plt.xlabel('Character')
plt.ylabel('Count')
plt.show()

### Sample Visualization

In [ ]:
fig, axes = plt.subplots(4, 9, figsize=(14, 7))
axes = axes.flatten()
for i, ax in enumerate(axes):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(CLASS_NAMES[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing
Normalize pixel values to `[0, 1]` and add a channel dimension.

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print('New training shape:', x_train.shape)

## 4. Model Architecture

In [ ]:
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, kernel_size=(3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, kernel_size=(3, 3), activation='relu'),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Model Training

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-4)

history = model.fit(
    x_train, y_train,
    epochs=12,
    batch_size=128,
    validation_split=0.1,
    callbacks=[early_stop, reduce_lr],
)

## 6. Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

### Training History

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')
plt.show()

### Confusion Matrix and Classification Report

In [ ]:
y_pred = np.argmax(model.predict(x_test, batch_size=256, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 7. Save Model

In [ ]:
model_dir = ROOT / 'model'
model_dir.mkdir(exist_ok=True)
save_path = model_dir / 'handwritten_character_cnn.keras'
model.save(save_path)
print('Saved', save_path)